In [2]:
import pandas as pd
from datetime import datetime
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import warnings
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [7]:
schema_Bases = [
        bigquery.SchemaField("CERTIFICADO_BANCO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_ALTA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_BAJA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("DIFERENCIA_DIAS", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("GLOSA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMERO_OPERACION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_OPERACION", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("ORIGEN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CUENTA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FUENTE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMERO_RECLAMO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CIERRE", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("NOMBRE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_BASE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.DATE)
]
def normalizar_dataframe(df, schema):
    columnas = [field.name for field in schema]
    
    for col in columnas:
        if col not in df.columns:
            df[col] = None
            
    return df[columnas]

### Archivo DAR

In [41]:
df_DAR= pd.read_excel("C:/data/AUTOMATIZACION - ANULACIONES/DAR/DAR 2025-Julio a Octubre.xlsx", 
                      sheet_name=0, dtype=str)
df_DAR.columns = (df_DAR.columns.str.strip()  # quitar espacios al inicio/fin
                   .str.upper()  # opcional: todo en mayúsculas
                   .str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/_ por _
)

In [42]:
df_DAR.drop(['CANAL','MES','CLIENTE','EMISI_N'], axis=1, inplace=True)
df_DAR= df_DAR.rename(columns={'CONTRATO':'CERTIFICADO_BANCO', 'PRD':'CODIGO_PRODUCTO', 'FEC_ALTA':'FECHA_ALTA', 
                               'FEC_BAJA':'FECHA_BAJA', 'DIAS':'DIFERENCIA_DIAS','DIV':'MONEDA'})

In [43]:
df_DAR['FECHA_ALTA']= pd.to_datetime(df_DAR['FECHA_ALTA'],format='%Y-%m-%d', errors='coerce').dt.date
df_DAR['FECHA_BAJA']= pd.to_datetime(df_DAR['FECHA_BAJA'],format='%Y-%m-%d', errors='coerce').dt.date
df_DAR['DIFERENCIA_DIAS'] = (pd.to_datetime(df_DAR['FECHA_BAJA']) - pd.to_datetime(df_DAR['FECHA_ALTA'])).dt.days
df_DAR['PRIMA'] = pd.to_numeric(df_DAR['PRIMA'], errors="coerce").astype('float64')

In [ ]:
df_DAR['CERTIFICADO_BANCO'] = (df_DAR['CERTIFICADO_BANCO'].str[:8] + df_DAR['CERTIFICADO_BANCO'].str[10:])

In [40]:
df_DAR = normalizar_dataframe(df_DAR, schema_Bases)

In [45]:
df_DAR.head(3)

,CERTIFICADO_BANCO,CODIGO_PRODUCTO,PRODUCTO,FECHA_ALTA,FECHA_BAJA,DIFERENCIA_DIAS,MONEDA,PRIMA,CERTIFICADO_BANCO2
0,00110118704000429349,800,SEGURO DE VIDA ...,2025-07-11,2025-07-15,4,USD,32.0,001101184000429349
1,00110002324000089792,800,SEGURO DE VIDA ...,2025-07-15,2025-07-15,0,USD,12.0,001100024000089792
2,00117794504006187952,803,SEG.CONT.PROTECCION ...,2025-07-02,2025-07-04,2,PEN,35.0,001177944006187952


In [9]:
df_DAR.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7024 entries, 0 to 7023
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CERTIFICADO_BANCO  7024 non-null   object 
 1   CODIGO_PRODUCTO    7024 non-null   object 
 2   PRODUCTO           7024 non-null   object 
 3   FECHA_ALTA         7024 non-null   object 
 4   FECHA_BAJA         7024 non-null   object 
 5   DIFERENCIA_DIAS    7024 non-null   int64  
 6   MONEDA             7024 non-null   object 
 7   PRIMA              7024 non-null   float64
dtypes: float64(1), int64(1), object(6)
memory usage: 439.1+ KB


### Archivo FCR1

In [13]:
df_FCR1= pd.read_excel("C:/data/AUTOMATIZACION - ANULACIONES/FCR1/(Del 11.11.25 al 17.11.25) NACAR.xlsx", sheet_name=0, dtype=str)

In [14]:
df_FCR1.columns = (df_FCR1.columns.str.strip()  # quitar espacios al inicio/fin
                   .str.upper()  # opcional: todo en mayúsculas
                   .str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/_ por _
)
df_FCR1 = df_FCR1.loc[:, ~df_FCR1.columns.duplicated()]

In [15]:
df_FCR1= df_FCR1.rename(columns={'N_MERO_DE_CONTRATO_DE_SEGURO':'CERTIFICADO_BANCO', 'TIPO_DE_SEGURO':'PRODUCTO', 'DIVISA':'MONEDA', 
                               'IMPORTE_ORIGINAL':'PRIMA'})

In [16]:
df_FCR1= df_FCR1[['CERTIFICADO_BANCO','PRODUCTO','MONEDA','PRIMA','GLOSA']]

In [18]:
df_FCR1.loc[df_FCR1['MONEDA'].str.strip().str.lower() == 'sol', 'MONEDA'] = 'PEN'
df_FCR1.loc[df_FCR1['MONEDA'].str.strip().str.lower() == 'dolar', 'MONEDA'] = 'USD'

In [19]:
df_FCR1['CERTIFICADO_BANCO'] = (df_FCR1['CERTIFICADO_BANCO'].str.strip().str.replace('-', '', regex=False))
df_FCR1['PRIMA'] = pd.to_numeric(df_FCR1['PRIMA'], errors="coerce").astype('float64')

In [20]:
df_FCR1.head(3)

,CERTIFICADO_BANCO,PRODUCTO,MONEDA,PRIMA,GLOSA
0,00117794524006461582,PROTECCIÓN MÚLTIPLE,PEN,35.0,ST-95577-PM-7794524006461582
1,00110368804000431564,MULTIRIESGO NEGOCIO,PEN,948.0,ST-95581-MN-0368804000431564
2,00117794564006471065,RENTA HOSPITALARIA,PEN,32.0,ST-95615-RH-7794564006471065


In [127]:
df_FCR1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 143 entries, 0 to 142
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CERTIFICADO_BANCO  143 non-null    object 
 1   PRODUCTO           143 non-null    object 
 2   MONEDA             143 non-null    object 
 3   PRIMA              143 non-null    float64
 4   GLOSA              143 non-null    object 
dtypes: float64(1), object(4)
memory usage: 5.7+ KB


### Archivo FCR2

In [136]:
df_FCR2= pd.read_excel("C:/data/AUTOMATIZACION - ANULACIONES/FCR2/0756_20.01.2026 al 22.01.2026.xlsx", sheet_name=0, dtype=str)
df_FCR2.columns = (df_FCR2.columns.str.strip()  # quitar espacios al inicio/fin
                   .str.upper()  # opcional: todo en mayúsculas
                   .str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/_ por _
)

In [137]:
df_FCR2= df_FCR2.rename(columns={'NRO__CONTRATO':'CERTIFICADO_BANCO', 'DIVISA':'MONEDA', 
                                 'SALDO':'PRIMA', 'GLOSA_1':'GLOSA', 'NUMERO_DE_CONTROL':'NUMERO_OPERACION'})

In [138]:
df_FCR2= df_FCR2[['CERTIFICADO_BANCO','MONEDA','PRIMA','GLOSA','NUMERO_OPERACION']]
df_FCR2['CERTIFICADO_BANCO'] = (df_FCR2['CERTIFICADO_BANCO'].str.strip().str.replace('-', '', regex=False))
df_FCR2['PRIMA'] = pd.to_numeric(df_FCR2['PRIMA'], errors="coerce").astype('float64')

In [139]:
df_FCR2.head(3)

,CERTIFICADO_BANCO,MONEDA,PRIMA,GLOSA,NUMERO_OPERACION
0,00110266434001167793,PEN,300.05,DAPST-99372-SATA-0266434001167793,NaN
1,00110982694000755163,PEN,200.55,DAPST-99376-SATA-0982694000755163,NaN
2,00110222784000752056,PEN,262.73,DAPST-99378-SATA-0222784000752056,NaN


In [140]:
df_FCR2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91 entries, 0 to 90
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CERTIFICADO_BANCO  89 non-null     object 
 1   MONEDA             91 non-null     object 
 2   PRIMA              91 non-null     float64
 3   GLOSA              91 non-null     object 
 4   NUMERO_OPERACION   2 non-null      object 
dtypes: float64(1), object(4)
memory usage: 3.7+ KB


### Archivo OMX

In [22]:
df_OMX= pd.read_excel("C:/data/AUTOMATIZACION - ANULACIONES/OMX/OMX Diciembre 2025.xlsx", sheet_name=0, dtype=str)
df_OMX.columns = (df_OMX.columns.str.strip()  # quitar espacios al inicio/fin
                   .str.upper()  # todo en mayúsculas
                   .str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra o número por _
)

In [23]:
def renombrar_col_duplicadas(columnas):
    contador = {}
    nuevas_columnas = []
    for col in columnas:
        if col in contador:
            contador[col] += 1
            nuevas_columnas.append(f"{col}_{contador[col]}")    
        else:
            contador[col] = 1
            nuevas_columnas.append(col)
    return nuevas_columnas

In [24]:
df_OMX.columns = renombrar_col_duplicadas(df_OMX.columns)
df_OMX = df_OMX.drop(columns=['PRODUCTO'])

In [25]:
df_OMX= df_OMX.rename(columns={'CERTIFICADO_BCO_CON_D_GITOS_DE_CONTROL':'CERTIFICADO_BANCO', 
                               'MONTO':'PRIMA', 'DETALLE':'GLOSA', 'N__DE_OPERACION':'NUMERO_OPERACION',
                               '_RECLAMO':'NUMERO_RECLAMO','PRODUCTO_2':'PRODUCTO'})

In [26]:
df_OMX['PRIMA'] = pd.to_numeric(df_OMX['PRIMA'], errors="coerce").astype('float64')
df_OMX['FECHA_OPERACION'] = pd.to_datetime(df_OMX['FECHA_OPERACION'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date

In [27]:
df_OMX= df_OMX[['CERTIFICADO_BANCO','PRODUCTO','MONEDA','PRIMA','GLOSA','NUMERO_OPERACION', 
                'FECHA_OPERACION','ORIGEN','CUENTA','FUENTE','NUMERO_RECLAMO']]

In [29]:
df_OMX.head(3)

,CERTIFICADO_BANCO,PRODUCTO,MONEDA,PRIMA,GLOSA,NUMERO_OPERACION,FECHA_OPERACION,ORIGEN,CUENTA,FUENTE,NUMERO_RECLAMO
0,00110178184000430476,SEGURO PARA VEHICULOS-LIMA,USD,183.00,OMXP75693 DEV01781840004304,2177916,2025-12-05,CONCILIACIÓN,BBVA 2265,LA/RI EMERGENTE - PAGO POR RECAUDO,NaN
1,00110229244001077725,SEGURO PARA INMUEBLES,USD,709.82,OMXP75693 DEV02292440010777,2178482,2025-12-30,CONCILIACIÓN,BBVA 2265,LA/RI EMERGENTE - PAGO POR RECAUDO,NaN
2,00110281384001016037,SEGURO PARA VEHICULOS-LIMA,USD,106.35,OMXP75693 DEV02813840010160,2178081,2025-12-13,CONCILIACIÓN,BBVA 2265,LA/RI EMERGENTE - PAGO POR RECAUDO,NaN


In [108]:
df_OMX.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CERTIFICADO_BANCO  235 non-null    object 
 1   PRODUCTO           235 non-null    object 
 2   MONEDA             252 non-null    object 
 3   PRIMA              252 non-null    float64
 4   GLOSA              252 non-null    object 
 5   NUMERO_OPERACION   252 non-null    object 
 6   FECHA_OPERACION    252 non-null    object 
 7   ORIGEN             252 non-null    object 
 8   CUENTA             252 non-null    object 
 9   FUENTE             252 non-null    object 
 10  NUMERO_RECLAMO     170 non-null    object 
dtypes: float64(1), object(10)
memory usage: 21.8+ KB


### Archivo PIC

In [64]:
df_PIC= pd.read_excel("C:/data/AUTOMATIZACION - ANULACIONES/PIC/(DEL 13.01.26 al 19.01.26) PIC (1).xlsx", 
                           sheet_name=0, dtype=str)
df_PIC.columns = (df_PIC.columns.str.strip()  # quitar espacios al inicio/fin
                   .str.upper()  # todo en mayúsculas
                   .str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra o número por _
)


In [65]:
df_PIC = df_PIC.loc[:, ~df_PIC.columns.duplicated()]

In [66]:
df_PIC= df_PIC.rename(columns={'N_MERO_DE_CONTRATO_DE_SEGURO':'CERTIFICADO_BANCO', 
                               'TIPO_DE_SEGURO':'PRODUCTO', 'DIVISA':'MONEDA', 
                               'IMPORTE_ORIGINAL':'PRIMA'})
df_PIC= df_PIC[['CERTIFICADO_BANCO','PRODUCTO','MONEDA','PRIMA','GLOSA']]

In [67]:
df_PIC['CERTIFICADO_BANCO'] = (df_PIC['CERTIFICADO_BANCO'].str.strip().str.replace('-', '', regex=False))
df_PIC['CERTIFICADO_BANCO'] = (df_PIC['CERTIFICADO_BANCO'].str[:8] + df_PIC['CERTIFICADO_BANCO'].str[10:])
df_PIC['PRODUCTO'] = df_PIC['PRODUCTO'].str.upper()
df_PIC['PRIMA'] = pd.to_numeric(df_PIC['PRIMA'], errors="coerce").astype('float64')
df_PIC.loc[df_PIC['MONEDA'].str.strip().str.lower() == 'sol', 'MONEDA'] = 'PEN'
df_PIC.loc[df_PIC['MONEDA'].str.strip().str.lower() == 'dolar', 'MONEDA'] = 'USD'

In [68]:
df_PIC.head(3)

,CERTIFICADO_BANCO,PRODUCTO,MONEDA,PRIMA,GLOSA
0,001107514000258244,PROTECCIÓN DE TARJETA ROYAL (NUEVO PT ROYAL),PEN,179.01,ST-98970-PTR-0751594000258244
1,001103484000578841,PROTECCIÓN DE TARJETA ROYAL (NUEVO PT ROYAL),PEN,179.01,ST-98972-PTR-0348054000578841
2,001102104001315800,PROTECCIÓN DE TARJETA ROYAL (NUEVO PT ROYAL),PEN,179.01,ST-98976-PTR-0210274001315800


In [69]:
df_PIC.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CERTIFICADO_BANCO  96 non-null     object 
 1   PRODUCTO           96 non-null     object 
 2   MONEDA             96 non-null     object 
 3   PRIMA              96 non-null     float64
 4   GLOSA              96 non-null     object 
dtypes: float64(1), object(4)
memory usage: 3.9+ KB


### Archivo RECLAMOS

In [42]:
df_RECLAMOS= pd.read_excel("C:/data/AUTOMATIZACION - ANULACIONES/RECLAMOS/CUADRO DE OBLIGACIONES 2026_ PRIMER TRIMESTRE.xlsx", 
                           sheet_name='DEVOLUCIONES GENERALES', dtype=str)
df_RECLAMOS.columns = (df_RECLAMOS.columns.str.strip()  # quitar espacios al inicio/fin
                   .str.upper()  # todo en mayúsculas
                   .str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra o número por _
)

In [44]:
df_RECLAMOS= df_RECLAMOS.rename(columns={'NRO_CONTRATO__CERTIFICADO_BANCO_':'CERTIFICADO_BANCO', 
                               'TIPO_DE_MONEDA_DE_LA_CUENTA_BANCARIA':'MONEDA', 
                               'MONTO_RECLAMADO_S__USD':'PRIMA', 
                               'FECHA_DE_SOLICITUD_A_OPERACIONES':'FECHA_OPERACION',
                               'FECHA_DE_CIERRE':'FECHA_CIERRE','RECLAMO':'NUMERO_RECLAMO'})

In [45]:
df_RECLAMOS['PRODUCTO'] = df_RECLAMOS['PRODUCTO'].str.upper()
df_RECLAMOS['PRIMA'] = df_RECLAMOS['PRIMA'].replace({',': ''}, regex=True)
df_RECLAMOS['PRIMA'] = pd.to_numeric(df_RECLAMOS['PRIMA'], errors="coerce").astype('float64')
df_RECLAMOS['FECHA_OPERACION'] = pd.to_datetime(df_RECLAMOS['FECHA_OPERACION'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_RECLAMOS['FECHA_CIERRE'] = pd.to_datetime(df_RECLAMOS['FECHA_CIERRE'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_RECLAMOS.loc[df_RECLAMOS['MONEDA'].str.strip().str.lower() == 'sol', 'MONEDA'] = 'PEN'
df_RECLAMOS.loc[df_RECLAMOS['MONEDA'].str.strip().str.lower() == 'dolar', 'MONEDA'] = 'USD'

In [46]:
df_RECLAMOS= df_RECLAMOS[['CERTIFICADO_BANCO','PRODUCTO','MONEDA','PRIMA','FECHA_OPERACION', 
                          'NUMERO_RECLAMO','FECHA_CIERRE']]

In [47]:
df_RECLAMOS.head(3)

,CERTIFICADO_BANCO,PRODUCTO,MONEDA,PRIMA,FECHA_OPERACION,NUMERO_RECLAMO,FECHA_CIERRE
0,00110354824000294250,VEHICULAR OPTATIVO,USD,870.45,2026-01-05,24122500504,2026-01-02
1,00110716814000247004,SALUD A TU ALCANCE,PEN,1991.00,2026-01-05,24122500762,2026-01-02
2,00110333264000574828,HOGAR TOTAL,PEN,1553.80,2026-01-05,23122501117,2026-01-02


In [62]:
df_RECLAMOS.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2785 entries, 0 to 2784
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CERTIFICADO_BANCO  2785 non-null   object 
 1   PRODUCTO           2785 non-null   object 
 2   MONEDA             2785 non-null   object 
 3   PRIMA              2785 non-null   float64
 4   FECHA_OPERACION    2785 non-null   object 
 5   NUMERO_RECLAMO     2785 non-null   object 
 6   FECHA_CIERRE       2785 non-null   object 
dtypes: float64(1), object(6)
memory usage: 152.4+ KB


### Archivo TICKET INFO

In [37]:
df_Ticket_info= pd.read_excel("C:/data/AUTOMATIZACION - ANULACIONES/TICKET INFORMACIÓN/1361434 - TICKET Y LÓGICAS.xlsx", 
                           sheet_name=0, dtype=str)
df_Ticket_info.columns = (df_Ticket_info.columns.str.strip().str.upper().
                       str.replace(r'[^A-Za-z0-9]', '_', regex=True))  # reemplazar todo lo que no sea letra o número por _

In [38]:
df_Ticket_info= df_Ticket_info.rename(columns={'NRO_CERT_BANCO':'CERTIFICADO_BANCO', 'TIPO_SEGURO':'PRODUCTO', 
                                'FEC_INI_VIG_CERT':'FECHA_INICIO_VIGENCIA', 'FEC_FIN_VIG_CERT':'FECHA_FIN_VIGENCIA',
                                'FECVE_COB':'FECHA_VE_COB', 'FECVE_COB_1':'FECHA_VE_COB_1','FECOCURR':'FECHA_OCURR',
                                'DOCUMENTO_COB':'DOC_COB', 'DOCUMENTO_ESTADO_COB':'DOC_ESTADO_COB',
                                'DOCUMENTO_ANU_INC':'DOC_ANU_INC','DOCUMENTO_ESTADO_ANU_INC':'DOC_ESTADO_ANU_INC',
                                'P_BRUTA':'PRIMA_BRUTA','MTO_LA':'MONTO_LA'})

In [39]:
df_Ticket_info['FECHA_INICIO_VIGENCIA'] = pd.to_datetime(df_Ticket_info['FECHA_INICIO_VIGENCIA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_Ticket_info['FECHA_FIN_VIGENCIA'] = pd.to_datetime(df_Ticket_info['FECHA_FIN_VIGENCIA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_Ticket_info['FECHA_INI_ULT_COBERTURA'] = pd.to_datetime(df_Ticket_info['FECHA_INI_ULT_COBERTURA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_Ticket_info['FECHA_FIN_ULT_COBERTURA'] = pd.to_datetime(df_Ticket_info['FECHA_FIN_ULT_COBERTURA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_Ticket_info['FECHA_VE_COB'] = pd.to_datetime(df_Ticket_info['FECHA_VE_COB'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_Ticket_info['FECHA_VE_COB_1'] = pd.to_datetime(df_Ticket_info['FECHA_VE_COB_1'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_Ticket_info['FECHA_OCURR'] = pd.to_datetime(df_Ticket_info['FECHA_OCURR'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_Ticket_info['CAN_LQ_COB'] = pd.to_numeric(df_Ticket_info['CAN_LQ_COB'], errors='coerce').astype('Int64')
df_Ticket_info['CAN_LQ_INC'] = pd.to_numeric(df_Ticket_info['CAN_LQ_INC'], errors='coerce').astype('Int64')
df_Ticket_info['CAN_LQ_ANU'] = pd.to_numeric(df_Ticket_info['CAN_LQ_ANU'], errors='coerce').astype('Int64')
df_Ticket_info['CAN_LQ_EMI'] = pd.to_numeric(df_Ticket_info['CAN_LQ_EMI'], errors='coerce').astype('Int64')
df_Ticket_info['PRIMA_COB'] = pd.to_numeric(df_Ticket_info['PRIMA_COB'], errors="coerce").astype('float64')
df_Ticket_info['PRIMA_INC_ANU'] = pd.to_numeric(df_Ticket_info['PRIMA_INC_ANU'], errors="coerce").astype('float64')
df_Ticket_info['PRIMA_BRUTA'] = pd.to_numeric(df_Ticket_info['PRIMA_BRUTA'], errors="coerce").astype('float64')
df_Ticket_info['MONTO_LA'] = pd.to_numeric(df_Ticket_info['MONTO_LA'], errors="coerce").astype('float64')
df_Ticket_info['MTOTOTRES'] = pd.to_numeric(df_Ticket_info['MTOTOTRES'], errors="coerce").astype('float64')
df_Ticket_info['MTOTOTAPROB'] = pd.to_numeric(df_Ticket_info['MTOTOTAPROB'], errors="coerce").astype('float64')
df_Ticket_info['MTOTOTPAGADO'] = pd.to_numeric(df_Ticket_info['MTOTOTPAGADO'], errors="coerce").astype('float64')
df_Ticket_info['MTOTOTPEND'] = pd.to_numeric(df_Ticket_info['MTOTOTPEND'], errors="coerce").astype('float64')

In [40]:
df_Ticket_info.head(2)

,IDEPOL,CERT_BANCO_EXCEL,CERTIFICADO_BANCO,PRODUCTO,CODPROD,NUMPOL,CLIENTE,NUMCERT,STSCERT,FECHA_INICIO_VIGENCIA,FECHA_FIN_VIGENCIA,FECHA_INI_ULT_COBERTURA,FECHA_FIN_ULT_COBERTURA,CAN_LQ_COB,CAN_LQ_INC,CAN_LQ_ANU,CAN_LQ_EMI,DOC_COB,DOC_ESTADO_COB,PRIMA_COB,FECHA_VE_COB,DOC_ANU_INC,DOC_ESTADO_ANU_INC,PRIMA_INC_ANU,FECHA_VE_COB_1,PRIMA_BRUTA,TIPODOC,NUMERO_LA,MONTO_LA,ESTADO_LA,CAN_LA_EMI,NUMSIN,STSSIN,FECHA_OCURR,MTOTOTRES,MTOTOTAPROB,MTOTOTPAGADO,MTOTOTPEND,DESCRAMO
0,8412329,00110002304000062010,00110002304000062010,PROTECCIﾓN MﾚLTIPLE,1103,500076,GODOFREDO NARCISO SINCHE MAYORCA,82244,ANU,2018-07-24,2026-11-22,2020-08-13,2020-09-24,26,0,0,26,646316329,COB,2.99,2018-08-27,NaN,NaN,NaN,NaT,2.46,LA,176505368,1.12,ACT,0,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN
1,8505629,00110002304000094737,00110002304000094737,PROTECCIﾓN DE TARJETA,1103,500077,HECTOR MARIANO CHAVEZ PEREYRA,43331,ANU,2019-08-09,2046-03-13,2021-08-09,2021-08-09,2,0,0,2,734473871,COB,30.00,2019-10-11,NaN,NaN,NaN,NaT,24.68,NaN,NaN,NaN,NaN,0,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN


In [41]:
df_Ticket_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121519 entries, 0 to 121518
Data columns (total 39 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   IDEPOL                   77971 non-null   object 
 1   CERT_BANCO_EXCEL         121519 non-null  object 
 2   CERTIFICADO_BANCO        77971 non-null   object 
 3   PRODUCTO                 121519 non-null  object 
 4   CODPROD                  77971 non-null   object 
 5   NUMPOL                   77971 non-null   object 
 6   CLIENTE                  77971 non-null   object 
 7   NUMCERT                  77971 non-null   object 
 8   STSCERT                  77971 non-null   object 
 9   FECHA_INICIO_VIGENCIA    77968 non-null   object 
 10  FECHA_FIN_VIGENCIA       77946 non-null   object 
 11  FECHA_INI_ULT_COBERTURA  77971 non-null   object 
 12  FECHA_FIN_ULT_COBERTURA  77971 non-null   object 
 13  CAN_LQ_COB               121519 non-null  Int64  
 14  CAN_

In [ ]:
schema_Ticket_Info = [
        bigquery.SchemaField("IDEPOL", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CERT_BANCO_EXCEL", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CERTIFICADO_BANCO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODPROD", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMPOL", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CLIENTE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMCERT", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("STSCERT", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_INICIO_VIGENCIA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_FIN_VIGENCIA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_INI_ULT_COBERTURA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_FIN_ULT_COBERTURA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("CAN_LQ_COB", bigquery.enums.SqlTypeNames.INT64),
        bigquery.SchemaField("CAN_LQ_INC", bigquery.enums.SqlTypeNames.INT64),
        bigquery.SchemaField("CAN_LQ_ANU", bigquery.enums.SqlTypeNames.INT64),
        bigquery.SchemaField("CAN_LQ_EMI", bigquery.enums.SqlTypeNames.INT64),
        bigquery.SchemaField("DOC_COB", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DOC_ESTADO_COB", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA_COB", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("FECHA_VE_COB", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("DOC_ANU_INC", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("DOC_ESTADO_ANU_INC", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA_INC_ANU", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("FECHA_VE_COB_1", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("PRIMA_BRUTA", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("TIPODOC", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMERO_LA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MONTO_LA", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("ESTADO_LA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CAN_LA_EMI", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMSIN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("STSSIN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MTOTOTRES", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("MTOTOTAPROB", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("MTOTOTPAGADO", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("MTOTOTPEND", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("DESCRAMO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NOMBRE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.STRING),
]

---

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [2]:
import pandas as pd
from datetime import datetime
import time
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import zipfile
import os
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
from google.cloud import bigquery
from google.api_core.exceptions import NotFound, GoogleAPICallError

In [ ]:
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
FOLDER_PATH= "data_cpozo/tramas_DTC/2026/"
DATASET_ID = "produccion"
TABLE_ID= "TRAMAS_diarias_BBVA_2025"
TABLE_CONTROL_ID= "CONTROL_tramas_diarias"

In [ ]:
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

bucket = storage_client.bucket(BUCKET_NAME)
blob_errores = bucket.blob('data_entries/REPORTES DE ERRORES/tabla_errores2.xlsx')
df_errores= pd.read_excel(BytesIO(blob_errores.download_as_string()), sheet_name='Sheet1', dtype={'CODIGO_ERROR': str, 'IDEERROR': str})


def tabla_existe(table_ref):
    try:
      tabla = bigquery_client.get_table(table_ref)
      return True
    except NotFound:
        # La tabla no existe
        return False
    except GoogleAPICallError as e:
        # Otros errores de BigQuery (permisos, conexión, etc.)
        print(f"⚠️ Error al consultar BigQuery: {e}")
        raise


def buscar_archivo(filename_txt: str) -> bool:
    # Verificar si existe la tabla de control
    table_control_ref = bigquery_client.dataset(DATASET_ID).table(TABLE_CONTROL_ID)
    if not tabla_existe(table_control_ref):
        return False
    else:
        # Buscar archivo en la tabla de control
        query = f"""
            SELECT COUNT(*) AS count
            FROM `{table_control_ref}`
            WHERE ARCHIVO_TXT = @archivo_txt
        """
        job_config = bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("archivo_txt", "STRING", filename_txt)]
        )

        df = bigquery_client.query(query, job_config=job_config).to_dataframe()
        return df["count"].iloc[0] > 0
    

#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    job_config = bigquery.LoadJobConfig()
    if tabla_existe(table_ref):
        # Abre la tabla para agregar registros
        job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND
    else:
        tabla_tramas = bigquery.Table(table_ref, schema=schema)
        tabla_tramas = bigquery_client.create_table(tabla_tramas)
        job_config.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
        print(f'ℹ️ ----- Se ha creado la tabla: {table_id} en el dataset: {dataset_id} -----')
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    #print(f'----- REGISTROS AGREGADOS CORRECTAMENTE EN: {table_id} -------')
    return

############### PROCESO DE CARGA DE ARCHIVOS EXCEL A BIGQUERY ##########

bucket = storage_client.bucket(BUCKET_NAME)
blobs_excels = list(bucket.list_blobs(prefix=FOLDER_PATH))

for blob in blobs_excels:
  if blob.name.endswith(".xlsx"):
    print(f"⏳ CARGANDO ARCHIVO: {blob.name} ....")
    file = blob.download_as_string()

In [6]:
schema_Bases = [
        bigquery.SchemaField("CERTIFICADO_BANCO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_ALTA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_BAJA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("DIFERENCIA_DIAS", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("GLOSA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMERO_OPERACION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_OPERACION", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("ORIGEN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CUENTA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FUENTE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMERO_RECLAMO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CIERRE", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("NOMBRE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_BASE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.DATE)
]

schema_Control = [
        bigquery.SchemaField("ARCHIVO_TXT", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.DATE)
]

columnas_oficiales = [field.name for field in schema_Bases]
print(columnas_oficiales)

['CERTIFICADO_BANCO', 'CODIGO_PRODUCTO', 'PRODUCTO', 'FECHA_ALTA', 'FECHA_BAJA', 'DIFERENCIA_DIAS', 'MONEDA', 'PRIMA', 'GLOSA', 'NUMERO_OPERACION', 'FECHA_OPERACION', 'ORIGEN', 'CUENTA', 'FUENTE', 'NUMERO_RECLAMO', 'FECHA_CIERRE', 'NOMBRE_ARCHIVO', 'TIPO_BASE', 'FECHA_CARGA']
